# 📊 06 - Data Input and Output

Most of data science is about **loading data, exploring it, and saving results**. Notebook 05 did this by hand with `open()`, `csv`, and `json`. Here, **Pandas** does the heavy lifting: one line to read a file into a table (a **DataFrame**) and one line to write it back out, in almost any format.

## ✅ Learning Goals
- Read CSV files (local or from a URL) into a DataFrame, and handle common variations with `read_csv` options
- Quickly inspect a DataFrame's shape, columns, types, and statistics
- Write CSV files without the "extra index column" mistake
- Read and write JSON in the common layouts, including nested data from APIs
- Read and write Excel workbooks, including multiple sheets
- **Convert** between CSV, JSON, and Excel, and spot what can go wrong along the way

## 0. Setup
Every file this notebook creates goes into a **`06-OutputFiles`** folder, through the `OUT` variable (the same pattern as notebook 05). Upload that folder along with this notebook when you submit.

In [1]:
import pandas as pd
from pathlib import Path

OUT = Path('06-OutputFiles')
OUT.mkdir(exist_ok=True)

## 1. Reading CSV Files

In [2]:
# Create a small CSV for demo
data = """Name,Age,Score
Alice,23,90
Bob,25,85
Charlie,22,95
"""
(OUT / 'students.csv').write_text(data, encoding='utf-8')

# Read CSV into a DataFrame
df = pd.read_csv(OUT / 'students.csv')
df

,Name,Age,Score
0,Alice,23,90
1,Bob,25,85
2,Charlie,22,95


Compare this to notebook 05, where `csv.reader` gave you **strings** that you had to convert by hand. Pandas looks at each column and picks a type for you:

In [3]:
df.dtypes

Name       str
Age      int64
Score    int64
dtype: object

### Reading from a URL
`read_csv` can read straight from the web, which is how many public datasets are shared. (This needs an internet connection.)

In [4]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
penguins = pd.read_csv(url)
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


📶 **Offline?** If that cell fails, run `penguins = df` instead so the cells below still work (they'll just have less interesting output).

### Not Every CSV Looks the Same
Real files come with different separators, codes for missing values, and more columns or rows than you need. `read_csv` has options for all of these:

| Option | Use it when... | Example |
|---|---|---|
| `sep` | Columns are separated by something other than a comma | `sep=';'`, `sep='\t'` (tab) |
| `na_values` | Missing values are written as a code | `na_values=['N/A', '-', '?']` |
| `usecols` | You only need some of the columns | `usecols=['Name', 'Score']` |
| `nrows` | The file is huge and you just want a peek | `nrows=100` |
| `encoding` | You get a `UnicodeDecodeError` (common with files exported from Excel on Windows) | `encoding='latin-1'` |
| `dtype` | You need to force a column's type | `dtype={'Zip': str}` |

In [5]:
messy = """Name;Age;Score
Dana;24;88
Eli;N/A;92
Fay;21;-
"""
(OUT / 'messy.csv').write_text(messy, encoding='utf-8')

messy_df = pd.read_csv(OUT / 'messy.csv', sep=';', na_values=['N/A', '-'])
messy_df

,Name,Age,Score
0,Dana,24.0,88.0
1,Eli,NaN,92.0
2,Fay,21.0,NaN


The missing values show up as `NaN` ("not a number"), which is Pandas' marker for missing data. Notice that `Age` and `Score` also turned into decimals (`24.0`): a number column with a missing value is stored as a float. Module 06 covers cleaning these up.

✅ **Your Turn**: Create your own small CSV (3–5 rows, at least one number column) at `OUT / 'my_data.csv'` and read it into a DataFrame called `my_df`.

In [6]:
# Your turn here
import pandas as pd
my_data = """Product, Quantity, Price
Mobile,4,10264
Car,1,35689
Coffee Maker,3,150
"""
(OUT / 'my_data.csv').write_text(my_data, encoding='utf-8')

my_df = pd.read_csv(OUT / 'my_data.csv', sep=',', na_values=['N/A', '-'])
my_df


,Product,Quantity,Price
0,Mobile,4,10264
1,Car,1,35689
2,Coffee Maker,3,150


## 2. Exploring DataFrames

A few attributes answer the first questions you'll have about any new dataset:

In [7]:
print('shape:  ', penguins.shape)          # (rows, columns)
print('columns:', list(penguins.columns))

shape:   (344, 7)
columns: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']


In [8]:
# Look at the first rows
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


`.tail()` shows the **last** rows, and `.sample(5)` shows 5 **random** rows, which is often more representative than the first few.

In [9]:
penguins.tail()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE
343,Gentoo,Biscoe,49.9,16.1,213.0,5400.0,MALE


In [10]:
penguins.sample(5)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
30,Adelie,Dream,39.5,16.7,178.0,3250.0,FEMALE
152,Chinstrap,Dream,46.5,17.9,192.0,3500.0,FEMALE
137,Adelie,Dream,40.2,20.1,200.0,3975.0,MALE
141,Adelie,Dream,40.6,17.2,187.0,3475.0,MALE
306,Gentoo,Biscoe,43.4,14.4,218.0,4600.0,FEMALE


In [11]:
# Info about columns, data types, and missing values
penguins.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


In [12]:
# Summary statistics for the number columns
penguins.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,342.000000,342.000000,342.000000,342.000000
mean,43.921930,17.151170,200.915205,4201.754386
std,5.459584,1.974793,14.061714,801.954536
min,32.100000,13.100000,172.000000,2700.000000
25%,39.225000,15.600000,190.000000,3550.000000
50%,44.450000,17.300000,197.000000,4050.000000
75%,48.500000,18.700000,213.000000,4750.000000
max,59.600000,21.500000,231.000000,6300.000000


💡 Read `.info()`'s **Non-Null Count** column carefully. If it's lower than the number of rows, that column has missing values. In `penguins`, several columns have a few.

✅ **Your Turn**: Use `.shape`, `.head()`, `.info()`, and `.describe()` on `my_df`.

In [13]:
# Your turn here
my_df.shape

(3, 3)

In [14]:
my_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Product    3 non-null      str  
 1    Quantity  3 non-null      int64
 2    Price     3 non-null      int64
dtypes: int64(2), str(1)
memory usage: 204.0 bytes


In [15]:
my_df.head()

,Product,Quantity,Price
0,Mobile,4,10264
1,Car,1,35689
2,Coffee Maker,3,150


In [16]:
my_df.describe()

,Quantity,Price
count,3.000000,3.000000
mean,2.666667,15367.666667
std,1.527525,18310.944551
min,1.000000,150.000000
25%,2.000000,5207.000000
50%,3.000000,10264.000000
75%,3.500000,22976.500000
max,4.000000,35689.000000


## 3. Writing CSV Files

In [17]:
# Save the DataFrame to a new CSV
df.to_csv(OUT / 'students_copy.csv', index=False)

print((OUT / 'students_copy.csv').read_text(encoding='utf-8'))

Name,Age,Score
Alice,23,90
Bob,25,85
Charlie,22,95



### Why `index=False`?
Every DataFrame has an **index**, the 0, 1, 2... row labels on the left. If you leave out `index=False`, Pandas writes the index as an extra unnamed column. Read the file back and you get a junk `Unnamed: 0` column:

In [18]:
df.to_csv(OUT / 'students_with_index.csv')        # forgot index=False
pd.read_csv(OUT / 'students_with_index.csv')

,Unnamed: 0,Name,Age,Score
0,0,Alice,23,90
1,1,Bob,25,85
2,2,Charlie,22,95


✅ **Your Turn**: Save `my_df` to `OUT / 'my_data_copy.csv'` (without the index), then print the file's raw text to check it.

In [19]:
# Your turn here
my_df.to_csv(OUT / 'my_data.csv', index=False)

print((OUT / 'my_data.csv').read_text(encoding='utf-8'))

Product, Quantity, Price
Mobile,4,10264
Car,1,35689
Coffee Maker,3,150



## 4. JSON Files
JSON is the language of web APIs and config files. A DataFrame can be written as JSON in several layouts, which Pandas calls **orients**. The one you'll want most often is `orient='records'`: a list with one dictionary per row, which is how most APIs return data.

In [20]:
df.to_json(OUT / 'students.json', orient='records', indent=2)

print((OUT / 'students.json').read_text(encoding='utf-8'))

[
  {
    "Name":"Alice",
    "Age":23,
    "Score":90
  },
  {
    "Name":"Bob",
    "Age":25,
    "Score":85
  },
  {
    "Name":"Charlie",
    "Age":22,
    "Score":95
  }
]


In [21]:
pd.read_json(OUT / 'students.json')

,Name,Age,Score
0,Alice,23,90
1,Bob,25,85
2,Charlie,22,95


⚠️ **The default orient is not `'records'`.** Without it, a DataFrame is saved **column by column**, keyed by the index. It's valid JSON, but awkward for anyone else to use:

In [22]:
print(df.to_json(indent=2))

{
  "Name":{
    "0":"Alice",
    "1":"Bob",
    "2":"Charlie"
  },
  "Age":{
    "0":23,
    "1":25,
    "2":22
  },
  "Score":{
    "0":90,
    "1":85,
    "2":95
  }
}


### JSON Lines
Large datasets and log files often use **JSON Lines** (`.jsonl`): one complete JSON object per line, with no surrounding list. Use `lines=True` to read or write it:

In [23]:
df.to_json(OUT / 'students.jsonl', orient='records', lines=True)
print((OUT / 'students.jsonl').read_text(encoding='utf-8'))

pd.read_json(OUT / 'students.jsonl', lines=True)

{"Name":"Alice","Age":23,"Score":90}
{"Name":"Bob","Age":25,"Score":85}
{"Name":"Charlie","Age":22,"Score":95}



,Name,Age,Score
0,Alice,23,90
1,Bob,25,85
2,Charlie,22,95


### Nested JSON from APIs
API data is usually **nested**: dictionaries inside dictionaries. `pd.read_json` would leave the inner dictionaries crammed into single cells. `pd.json_normalize` **flattens** them into regular columns, joining the names with a dot:

In [24]:
import json

api_response = [
    {'id': 1, 'name': 'Alice', 'contact': {'email': 'alice@example.com', 'city': 'Dallas'}},
    {'id': 2, 'name': 'Bob',   'contact': {'email': 'bob@example.com',   'city': 'Wichita Falls'}},
]
(OUT / 'api_users.json').write_text(json.dumps(api_response, indent=2), encoding='utf-8')

with open(OUT / 'api_users.json', encoding='utf-8') as f:
    raw = json.load(f)

users = pd.json_normalize(raw)
users

,id,name,contact.email,contact.city
0,1,Alice,alice@example.com,Dallas
1,2,Bob,bob@example.com,Wichita Falls


✅ **Your Turn**: Save `my_df` to `OUT / 'my_data.json'` using `orient='records'` and `indent=2`. Print the file's text, then read it back with `pd.read_json`.

In [25]:
# Your turn here
my_df.to_json(OUT / 'my_data.jsonl', orient='records', lines=True)

print((OUT / 'my_data.jsonl').read_text(encoding='utf-8'))
pd.read_json(OUT / 'my_data.jsonl', lines=True)

{"Product":"Mobile"," Quantity":4," Price":10264}
{"Product":"Car"," Quantity":1," Price":35689}
{"Product":"Coffee Maker"," Quantity":3," Price":150}



,Product,Quantity,Price
0,Mobile,4,10264
1,Car,1,35689
2,Coffee Maker,3,150


## 5. Excel Files
Pandas reads and writes Excel through a separate package, `openpyxl`. Install it **once** with the `%pip` magic from notebook 01, which installs into this notebook's environment. Restart the kernel afterwards if Pandas still says it's missing.

In [26]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
# Save to Excel
df.to_excel(OUT / 'students.xlsx', index=False)

# Read the Excel file back
pd.read_excel(OUT / 'students.xlsx')

,Name,Age,Score
0,Alice,23,90
1,Bob,25,85
2,Charlie,22,95


### Workbooks with Several Sheets
Real workbooks often have several sheets. To **write** several, use `pd.ExcelWriter`. To **read** them, use `sheet_name`:
- `sheet_name='Grades'`: one sheet, by name (`sheet_name=0` picks the first)
- `sheet_name=None`: **every** sheet, returned as a dictionary of `{sheet name: DataFrame}`

In [28]:
with pd.ExcelWriter(OUT / 'school.xlsx') as writer:
    df.to_excel(writer, sheet_name='Grades', index=False)
    users.to_excel(writer, sheet_name='Contacts', index=False)

pd.read_excel(OUT / 'school.xlsx', sheet_name='Contacts')

,id,name,contact.email,contact.city
0,1,Alice,alice@example.com,Dallas
1,2,Bob,bob@example.com,Wichita Falls


In [29]:
all_sheets = pd.read_excel(OUT / 'school.xlsx', sheet_name=None)

for sheet_name, sheet_df in all_sheets.items():
    print(f'{sheet_name}: {sheet_df.shape[0]} rows, {sheet_df.shape[1]} columns')

Grades: 3 rows, 3 columns
Contacts: 2 rows, 4 columns


✅ **Your Turn**: Save `my_df` to `OUT / 'my_data.xlsx'` and read it back.

In [30]:
# Your turn here
# Save to Excel
my_df.to_excel(OUT / 'my_data.xlsx', index=False)
pd.read_excel(OUT / 'my_data.xlsx')

,Product,Quantity,Price
0,Mobile,4,10264
1,Car,1,35689
2,Coffee Maker,3,150


## 6. Converting Between Formats
Once data is in a DataFrame, converting it is always the same two steps: **read** with `pd.read_<format>()`, then **write** with `df.to_<format>()`.

| From \ To | CSV | JSON | Excel |
|---|---|---|---|
| **CSV** | | `read_csv` → `to_json` | `read_csv` → `to_excel` |
| **JSON** | `read_json` → `to_csv` | | `read_json` → `to_excel` |
| **Excel** | `read_excel` → `to_csv` | `read_excel` → `to_json` | |

### CSV → JSON

In [31]:
pd.read_csv(OUT / 'students.csv').to_json(OUT / 'converted_students.json', orient='records', indent=2)

print((OUT / 'converted_students.json').read_text(encoding='utf-8'))

[
  {
    "Name":"Alice",
    "Age":23,
    "Score":90
  },
  {
    "Name":"Bob",
    "Age":25,
    "Score":85
  },
  {
    "Name":"Charlie",
    "Age":22,
    "Score":95
  }
]


### JSON → CSV
For flat JSON, this is just `read_json` followed by `to_csv`. For **nested** JSON, flatten it with `json_normalize` first, because CSV can't hold a dictionary inside a cell:

In [32]:
users.to_csv(OUT / 'converted_users.csv', index=False)

print((OUT / 'converted_users.csv').read_text(encoding='utf-8'))

id,name,contact.email,contact.city
1,Alice,alice@example.com,Dallas
2,Bob,bob@example.com,Wichita Falls



### Excel → CSV (One File per Sheet)
CSV has no concept of "sheets", so a workbook becomes **one CSV per sheet**. Loop over the dictionary from `sheet_name=None`:

In [33]:
for sheet_name, sheet_df in pd.read_excel(OUT / 'school.xlsx', sheet_name=None).items():
    out_file = OUT / f'school_{sheet_name}.csv'
    sheet_df.to_csv(out_file, index=False)
    print('Wrote', out_file)

Wrote 06-OutputFiles\school_Grades.csv
Wrote 06-OutputFiles\school_Contacts.csv


### ⚠️ What Can Go Wrong in a Conversion
Formats don't store the same information, so a conversion can **quietly change your data**:
- **Leading zeros vanish.** CSV has no types, so `read_csv` turns ZIP code `'01234'` into the number `1234`. Fix it with `dtype={'Zip': str}`.
- **Formatting is lost.** Converting Excel → CSV drops colors, formulas (you get the computed values), and every sheet but one per file.
- **Nesting doesn't fit.** JSON → CSV needs `json_normalize`, or you get dictionaries crammed into cells.
- **The index sneaks in** whenever you forget `index=False`.

In [34]:
(OUT / 'zips.csv').write_text("City,Zip\nBoston,02108\nDallas,75201\n", encoding='utf-8')

print(pd.read_csv(OUT / 'zips.csv'))                        # 02108 became 2108!
print()
print(pd.read_csv(OUT / 'zips.csv', dtype={'Zip': str}))    # kept as text

     City    Zip
0  Boston   2108
1  Dallas  75201

     City    Zip
0  Boston  02108
1  Dallas  75201


✅ **Your Turn**: Round-trip your data. Read `OUT / 'my_data.json'` (from section 4), save it as `OUT / 'my_data_roundtrip.csv'`, read **that** back, and check whether it matches the original with `my_df.equals(...)`. If it doesn't match, compare the `.dtypes` of both to find out why.

In [35]:
# Your turn here

# Read the original JSON file
#my_df = pd.read_json(OUT / 'my_data.json')

# Save as CSV
my_df.to_csv(OUT / 'my_data_roundtrip.csv', index=False)

# Read the CSV back in
#roundtrip_df = pd.read_csv(OUT / 'my_data_roundtrip.csv')
roundtrip_df = pd.read_csv(
    OUT / 'my_data_roundtrip.csv',
    dtype={'Quantity': int}
)
# Check whether they match
print(my_df.equals(roundtrip_df))
print(my_df.dtypes)
print()
print(roundtrip_df.dtypes)


True
Product        str
 Quantity    int64
 Price       int64
dtype: object

Product        str
 Quantity    int64
 Price       int64
dtype: object


---
### Summary
- `pd.read_csv()` loads data into a DataFrame and picks column types for you. Use `sep`, `na_values`, `usecols`, `nrows`, `encoding`, and `dtype` for messy files. It also reads straight from a URL.
- `.shape`, `.columns`, `.dtypes`, `.head()`, `.info()`, and `.describe()` give you a quick first look.
- `df.to_csv(..., index=False)` saves results. Forgetting `index=False` gives you an `Unnamed: 0` column.
- For JSON, use `orient='records'` for a list of rows, `lines=True` for JSON Lines, and `pd.json_normalize()` to flatten nested API data.
- `to_excel`/`read_excel` need `openpyxl`. Use `pd.ExcelWriter` to write several sheets and `sheet_name=None` to read them all.
- Converting formats = `read_<format>` + `to_<format>`. Watch for lost leading zeros, lost formatting, nesting, and stray index columns.

## 🏋️ Practice

### Practice 1 — Select a Column (easy)
From `df`, print just the `'Score'` column, then print its average with `.mean()`. Do the same for the `'body_mass_g'` column of `penguins`. Does `.mean()` mind the missing values?

In [36]:
# Your code here
# Score column from df
print(df['Score'])
print("Average score:", df['Score'].mean())

# body_mass_g column from penguins
print(penguins['body_mass_g'])
print("Average body mass:", penguins['body_mass_g'].mean())

0    90
1    85
2    95
Name: Score, dtype: int64
Average score: 90.0
0      3750.0
1      3800.0
2      3250.0
3         NaN
4      3450.0
        ...  
339       NaN
340    4850.0
341    5750.0
342    5200.0
343    5400.0
Name: body_mass_g, Length: 344, dtype: float64
Average body mass: 4201.754385964912


### Practice 2 — Filter and Save (medium)
Filter `df` down to only students with a `Score` of 90 or above. Save that filtered DataFrame to `OUT / 'top_students.csv'` **and** to `OUT / 'top_students.json'` (as records).

In [37]:
top_students = df[df['Score'] >= 90]

# Save to CSV
top_students.to_csv(OUT / 'top_students.csv', index=False)

# Save to JSON as records
top_students.to_json(
    OUT / 'top_students.json',
    orient='records',
    indent=2
)

print(top_students)

      Name  Age  Score
0    Alice   23     90
2  Charlie   22     95


### Practice 3 — Read It Back and Verify (harder)
Read `OUT / 'top_students.csv'` back into a new DataFrame and confirm (with an `if`/`print`, not just by eyeballing it) that every row in it has a `Score` of 90 or above. *Hint:* `(new_df['Score'] >= 90).all()` is `True` only if **every** value passes. Then do the same check on `top_students.json`.

In [38]:
# Your code here

# Check CSV
check_df = pd.read_csv(OUT / 'top_students.csv')

if (check_df['Score'] >= 90).all():
    print("CSV Success: Every student has a Score of 90 or above.")
else:
    print("CSV Error: Some students have a Score below 90.")

# Check JSON
check_json_df = pd.read_json(OUT / 'top_students.json')

if (check_json_df['Score'] >= 90).all():
    print("JSON Success: Every student has a Score of 90 or above.")
else:
    print("JSON Error: Some students have a Score below 90.")


CSV Success: Every student has a Score of 90 or above.
JSON Success: Every student has a Score of 90 or above.


## 🔥 Challenge (Optional) — A Loader for Any Format
Combine what you learned in Module 04 (More Functions & Loops), especially `safe_read_grades` in the files notebook, with this notebook's readers. Write a function `load_data(filename)` that:
1. Checks `Path(filename).suffix` first, and returns `None` with a message like `"Sorry, .txt files aren't supported"` for anything other than `.csv`, `.json`, or `.xlsx`.
2. Otherwise uses the suffix to pick the right reader (`.csv` → `pd.read_csv`, `.json` → `pd.read_json`, `.xlsx` → `pd.read_excel`) and returns the DataFrame.
3. Catches `FileNotFoundError` and returns `None` with a friendly message if the file doesn't exist.

**Bonus:** write `convert(src, dst)` that uses `load_data` to read `src` and picks the right `to_<format>` method from `dst`'s extension.

In [39]:
def load_data(filename):
    # Your code here
    path = Path(filename)
    suffix = path.suffix.lower()

    if suffix not in ['.csv', '.json', '.xlsx']:
        print(f"Sorry, {suffix} files aren't supported.")
        return None

    try:
        if suffix == '.csv':
            return pd.read_csv(path)

        elif suffix == '.json':
            return pd.read_json(path)

        elif suffix == '.xlsx':
            return pd.read_excel(path)

    except FileNotFoundError:
        print(f"Sorry, '{filename}' was not found.")
        return None
    pass

load_data(OUT / 'students.csv')
load_data(OUT / 'students.json')
load_data(OUT / 'students.xlsx')
load_data(OUT / 'does_not_exist.csv')
load_data('notes.txt')

Sorry, '06-OutputFiles\does_not_exist.csv' was not found.
Sorry, .txt files aren't supported.


## 📦 Check Your Output Files
Run this last to see everything you created. All of it should be inside `06-OutputFiles`. Upload that folder along with this notebook.

In [40]:
for path in sorted(OUT.rglob('*')):
    print(path)

06-OutputFiles\api_users.json
06-OutputFiles\converted_students.json
06-OutputFiles\converted_users.csv
06-OutputFiles\messy.csv
06-OutputFiles\my_data.csv
06-OutputFiles\my_data.jsonl
06-OutputFiles\my_data.xlsx
06-OutputFiles\my_data_roundtrip.csv
06-OutputFiles\school.xlsx
06-OutputFiles\school_Contacts.csv
06-OutputFiles\school_Grades.csv
06-OutputFiles\students.csv
06-OutputFiles\students.json
06-OutputFiles\students.jsonl
06-OutputFiles\students.xlsx
06-OutputFiles\students_copy.csv
06-OutputFiles\students_with_index.csv
06-OutputFiles\top_students.csv
06-OutputFiles\top_students.json
06-OutputFiles\zips.csv
